# 02 — Order Feasibility & Lot Assembly

Exploratory notebook for DERA-ZN stages **D (Detect)** and **E (Evaluate)**: given a buyer order, which farmers combine into candidate consolidated lots, and are those lots feasible to deliver?

> **Ground rule:** the deterministic tools in `tools/` do all the math — this notebook imports `validate_data`, `assemble_lots`, `route_risk`, `evaluate`, and `score_grid` and only *explores* their output. No number is computed by hand here.

What we look at:
1. Pick an order and Detect it
2. Assemble the three candidate lots (cheapest / nearest / reliability)
3. Route & perishability risk per lot
4. The full Evaluate factor table
5. The zone-normalized grid ranking (a peek at stage **R**)
6. Feasibility sweep across *all* orders (incl. the blocked and over-cap ones)

In [1]:
import os, sys

TOOLS = os.path.abspath(os.path.join(os.getcwd(), "..", "tools"))
if TOOLS not in sys.path:
    sys.path.insert(0, TOOLS)

import assemble_lots
import config
import dataio
import evaluate as ev
import route_risk as rr
import score_grid as sg
import validate_data as vd

farmers = dataio.load_farmers()
orders = dataio.load_orders()
transport = dataio.load_transport()
weather = dataio.load_weather()


def show_table(headers, rows):
    rows = [tuple(str(c) for c in r) for r in rows]
    widths = [max([len(str(h))] + [len(r[i]) for r in rows]) for i, h in enumerate(headers)]
    line = " | ".join(str(h).ljust(w) for h, w in zip(headers, widths))
    print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(c.ljust(w) for c, w in zip(r, widths)))

print("loaded:", len(farmers), "farmers,", len(orders), "orders")

loaded: 26 farmers, 5 orders


## 1. The worked example: `order_001`

Change `ORDER_ID` below to explore a different order.

In [2]:
ORDER_ID = "order_001"
order = dataio.get_order(ORDER_ID, orders)

for k in ("buyer_name", "commodity", "quantity_kg", "quality_required", "max_price_per_kg_khr",
          "delivery_province", "delivery_commune", "delivery_date", "perishability_days"):
    print(f"{k:>22}: {order[k]}")

            buyer_name: Phnom Fresh Market
             commodity: bok_choy
           quantity_kg: 1500
      quality_required: B
  max_price_per_kg_khr: 3800
     delivery_province: Kandal
      delivery_commune: Ta Khmau
         delivery_date: 2026-07-22
    perishability_days: 3


In [3]:
detect = vd.build_detect(ORDER_ID, farmers, orders, transport, weather)
s = detect["supply"]
print("Detect verdict:", "CLEAN" if detect["clean"] else "BLOCKED")
print(f"qualifying supply: {s['qualifying_kg']} kg of {s['commodity']} at grade >= {s['quality_required']} "
      f"from {s['n_qualifying_farmers']} farmers (order needs {s['required_qty_kg']} kg)")
for b in detect["blockers"]:
    print("BLOCKER:", b["code"], "-", b["message"])

Detect verdict: CLEAN
qualifying supply: 2550 kg of bok_choy at grade >= B from 6 farmers (order needs 1500 kg)


## 2. Candidate consolidated lots

`assemble_lots.build_candidates` builds one lot per strategy (docs/business_rules.md §3): **cheapest_first**, **nearest_first**, **reliability_first**. Allocation is greedy and never exceeds a farmer's declared quantity — deciding *who earns* is deterministic code, not prose.

In [4]:
lots = assemble_lots.build_candidates(order, farmers, transport)
for lot in lots:
    print(f"\n=== {lot['variant']} — {lot['n_farmers']} farmers, {lot['total_kg']} kg, coverage {lot['coverage']} ===")
    show_table(("farmer", "zone", "grade", "kg", "ask_khr/kg", "reliability"),
               [(a["farmer_id"], a["zone"], a["grade"], a["allocated_kg"],
                 a["ask_price_per_kg_khr"], a["reliability_score"]) for a in lot["allocations"]])


=== cheapest_first — 4 farmers, 1500 kg, coverage 1.0 ===
farmer | zone                   | grade | kg  | ask_khr/kg | reliability
------------------------------------------------------------------------
F010   | Takeo:Doun Kaev        | B     | 400 | 2900       | 0.83       
F002   | Kampong Cham:Prek Koy  | B     | 350 | 3000       | 0.78       
F006   | Kandal:Kien Svay       | B     | 300 | 3100       | 0.71       
F001   | Kampong Cham:Kang Meas | A     | 450 | 3200       | 0.92       

=== nearest_first — 4 farmers, 1500 kg, coverage 1.0 ===
farmer | zone                   | grade | kg  | ask_khr/kg | reliability
------------------------------------------------------------------------
F005   | Kandal:Ta Khmau        | A     | 450 | 3400       | 0.88       
F006   | Kandal:Kien Svay       | B     | 300 | 3100       | 0.71       
F010   | Takeo:Doun Kaev        | B     | 400 | 2900       | 0.83       
F001   | Kampong Cham:Kang Meas | A     | 350 | 3200       | 0.92       

=== re

## 3. Route & perishability risk per lot

`route_risk.route_risk_for_lot` scores each lot 0–1 (higher = worse) using the approved *rainy-season perishable* sub-weights, and reports the tightest spoilage margin across the lot's legs.

In [5]:
rows = []
for lot in lots:
    r = rr.route_risk_for_lot(lot, order, transport, weather)
    rows.append((lot["variant"], f"{r['route_risk']:.3f}", f"{r['spoilage_margin_days']:.2f} d",
                 f"{r['worst_leg_risk']:.3f}", f"{r['weather']['temp_c']} C / {r['weather']['flood_risk']}"))
show_table(("variant", "route_risk", "spoilage_margin", "worst_leg", "delivery weather"), rows)

variant           | route_risk | spoilage_margin | worst_leg | delivery weather
-------------------------------------------------------------------------------
cheapest_first    | 0.236      | 2.90 d          | 0.303     | 33.0 C / medium 
nearest_first     | 0.201      | 2.92 d          | 0.303     | 33.0 C / medium 
reliability_first | 0.199      | 2.75 d          | 0.303     | 33.0 C / medium 


## 4. The full Evaluate factor table

`evaluate.build_evaluate` combines lots + 0%-cut pricing + route risk into the four raw decision factors the grid consumes.

In [6]:
result = ev.build_evaluate(order, farmers, transport, weather)
rows = []
for c in result["candidates"]:
    f = c["factors"]
    rows.append((c["variant"], f["coverage"], f"{f['headroom_per_kg_khr']:,}",
                 f"{f['route_risk']:.3f}", f"{f['spoilage_margin_days']:.2f}",
                 "feasible" if c["feasible"] else "INFEASIBLE"))
show_table(("variant", "coverage", "headroom khr/kg", "route_risk", "spoilage_margin d", "verdict"), rows)
print(f"\n{result['n_feasible']}/{result['n_candidates']} candidates feasible")

variant           | coverage | headroom khr/kg | route_risk | spoilage_margin d | verdict 
------------------------------------------------------------------------------------------
cheapest_first    | 1.0      | 226.7           | 0.236      | 2.90              | feasible
nearest_first     | 1.0      | 180.43          | 0.201      | 2.92              | feasible
reliability_first | 1.0      | 77.73           | 0.199      | 2.75              | feasible

3/3 candidates feasible


## 5. Peek at the Recommend stage — the zone-normalized grid

Each factor is min-max normalized **across the candidate lots** (the "ZN" step), then combined with the approved balanced weights (coverage .35 / price .30 / route .20 / freshness .15).

In [7]:
grid = sg.build_grid(result["candidates"])
rows = []
for r in grid["ranked"]:
    n = r["normalized"]
    rows.append((r["variant"], r["score"], n["coverage"], n["price"], n["route"], n["freshness"]))
show_table(("variant", "score", "n_coverage", "n_price", "n_route", "n_freshness"), rows)
print("\nwinner:", grid["winner"]["variant"], "— weights:", grid["weights"])

variant           | score  | n_coverage | n_price | n_route | n_freshness
-------------------------------------------------------------------------
nearest_first     | 0.8933 | 1.0        | 0.6894  | 0.9326  | 1.0        
cheapest_first    | 0.7824 | 1.0        | 1.0     | 0.0     | 0.8827     
reliability_first | 0.55   | 1.0        | 0.0     | 1.0     | 0.0        

winner: nearest_first — weights: {'coverage': 0.35, 'price': 0.3, 'route': 0.2, 'freshness': 0.15}


## 6. Feasibility sweep across all orders

Includes the two designed negative cases: `order_004` (volume-gap blocker) and `order_005` (buyer cap below cost).

In [8]:
rows = []
for o in orders:
    d = vd.build_detect(o["order_id"], farmers, orders, transport, weather)
    if not d["clean"]:
        rows.append((o["order_id"], o["commodity"], "BLOCKED at Detect",
                     ", ".join(b["code"] for b in d["blockers"]), "—"))
        continue
    res = ev.build_evaluate(o, farmers, transport, weather)
    g = sg.build_grid(res["candidates"])
    if g["winner"]:
        rows.append((o["order_id"], o["commodity"], f"{res['n_feasible']}/{res['n_candidates']} feasible",
                     "—", f"winner: {g['winner']['variant']} ({g['winner']['score']})"))
    else:
        rows.append((o["order_id"], o["commodity"], "0 feasible", "over buyer cap",
                     "CANNOT FULFILL — renegotiate"))
show_table(("order", "commodity", "evaluate", "reason", "recommendation"), rows)

order     | commodity     | evaluate          | reason         | recommendation                 
------------------------------------------------------------------------------------------------
order_001 | bok_choy      | 3/3 feasible      | —              | winner: nearest_first (0.8933) 
order_002 | morning_glory | 3/3 feasible      | —              | winner: cheapest_first (0.9903)
order_003 | cabbage       | 0 feasible        | over buyer cap | CANNOT FULFILL — renegotiate   
order_004 | long_bean     | BLOCKED at Detect | VOLUME_GAP     | —                              
order_005 | cucumber      | 0 feasible        | over buyer cap | CANNOT FULFILL — renegotiate   


## Takeaways

- The three assembly strategies produce genuinely different lots with a real tradeoff: cheapest-first maximizes price headroom, nearest-first minimizes route risk & spoilage, reliability-first is safest but most expensive.
- Zone-normalization makes those factors comparable in one grid; under the approved balanced weights, **nearest_first** wins `order_001`.
- The pipeline refuses correctly: `order_004` blocks on the volume gap, `order_005` has no lot under the buyer's cap.
- Everything here mirrors what `/run-dera <order_id>` does — plus, in the real pipeline, the Act stage and the always-on **audit gate**, ending with *human approval*.

**Next notebook:** [03_pricing_profit_split.ipynb](03_pricing_profit_split.ipynb) — the 0%-cut price build-up in detail.